In [1]:
# !uv add pydantic-ai

In [2]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

In [3]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

In [4]:
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {'evidently'}

highlighter = Highlighter(
    highlight_fields=['content'],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer='snowball', stop_words=stopwords)
)

In [5]:
file_index = {doc['filename']: doc['content'] for doc in parsed_docs}

In [6]:
from typing import Any, Dict, List


class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self,
        index: Any,
        highlighter: Any,
        file_index: Dict[str, str]
    ) -> None:
        """
        Initialize the SearchTools instance.

        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.

        Args:
            query (str): The search query to look up in the index.

        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(
            query=query,
            num_results=5
        )

        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.

        Args:
            filename (str): The filename of the file to retrieve.

        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"


In [7]:
search_tools = SearchTools(index, highlighter, file_index)

In [8]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2–3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2–3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [9]:
from pydantic_ai import Agent
from toyaikit.tools import get_instance_methods

In [10]:
# tools = [search_tools.search, search_tools.get_file]
tools = get_instance_methods(search_tools)

In [11]:

search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)
    

In [12]:
query = 'how do I use evidently to monitor my machine learning models?'

In [13]:
result = await search_agent.run(query)

In [14]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='how do I use evidently to monitor my machine learning models?', timestamp=datetime.datetime(2026, 3, 5, 21, 28, 40, 962533, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 5, 21, 28, 40, 962698, tzinfo=datetime.timezone.utc), instructions='You\'re a documentation assistant.\n\nAnswer the user question using only the documentation knowledge base.\n\nMake 3 iterations:\n\n1) First iteration:\n   - Perform one search using the search tool to identify potentially relevant documents.\n   - Explain (in 2–3 sentences) why this search query is appropriate for the user question.\n\n2) Second iteration:\n   - Analyze the results from the previous search.\n   - Based on the filenames or documents returned, perform:\n       - Up to 2 additional search queries to refine or expand coverage, and\n       - One or more get_file calls to retrieve the full content of the most relevant documents.\n   - For each search or get_file call, exp

In [15]:
print(result.output)

I retrieved the contents of two documents: "Overview" and "What is Evidently?". 

1. The "Overview" document details several methods to monitor ML models, focusing on batch monitoring and tracing with scheduled evaluations. It explains that batch monitoring is ideal for ML pipelines and regular evaluations, while tracing allows for capturing comprehensive data from applications for more frequent assessments. This foundational knowledge is critical for your understanding of the available monitoring features.

2. The second document provides a general introduction to Evidently, describing it as an open-source library that facilitates evaluation, testing, and monitoring of AI systems. It highlights its capabilities, such as over 100 evaluation metrics and user-friendly tools for collaboration and observation.

In summary, to monitor your machine learning models using Evidently, you can either set up batch monitoring jobs to periodically evaluate model performance and data quality or imple

In [17]:
messages = result.all_messages()

In [18]:
result.usage()

RunUsage(input_tokens=4051, cache_read_tokens=1152, output_tokens=509, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, requests=3, tool_calls=3)

In [19]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

request
USER: how do I use evidently to monitor my machine learning models?

response
TOOL CALL: search {"query":"monitor machine learning models"}

request
TOOL RETURN: search

response
I performed a search using the query "monitor machine learning models" to identify relevant documents that could provide information about monitoring ML models with Evidently. This query is appropriate as it directly targets the user's inquiry about monitoring ML models and is likely to yield documents related to the functionality, setup, and best practices for model monitoring.

Next, I will analyze the results from the search. The documents returned include an overview of production AI quality monitoring, options for batch monitoring jobs, and general information about model evaluation and monitoring. I will retrieve the document titled "Overview," as it likely contains crucial information on how to set up monitoring systems effectively. Additionally, I will check if there are more details in the "Wh

In [20]:
new_prompt = 'show me the code'

In [21]:
result2 = await search_agent.run(new_prompt, message_history=messages)

In [22]:
print(result2.output)

I retrieved the document titled "Tutorials and guides," which contains valuable information about various code examples and workflows for using Evidently to monitor machine learning models.

Here are some key points regarding the code examples available:

1. **Quickstarts**: There are quickstart guides that can help you get started with your monitoring setup:
   - **ML quickstart**: This guide focuses on testing tabular data quality and detecting data drift.
   - **Tracing quickstart**: This guide explains how to collect inputs and outputs from your application for monitoring.

2. **Integration examples**: Several integration examples illustrate how to run Evidently evaluations within CI/CD workflows, use different LLM providers, and visualize evaluation metrics with Grafana.

3. **Specific workflows**: Various tutorials are provided for both LLM and ML evaluations, focusing on different aspects of machine learning performance monitoring, such as model quality and data drift. Each tuto

In [23]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [24]:
print_messages(result2.new_messages())

request
USER: show me the code

response
TOOL CALL: search {"query":"code examples monitoring machine learning models"}

request
TOOL RETURN: search

response
I performed a search using the query "code examples monitoring machine learning models" to locate specific documentation that includes code snippets for implementing monitoring features in Evidently. This query is suitable as it directly addresses the user's need for programming examples relevant to monitoring ML models.

Next, I will analyze the search results. Among the documents returned, the "Tutorials and guides" entry appears to specifically mention end-to-end code examples related to workflows and might include the information you're seeking. I will retrieve the contents of this document as it should provide practical examples to demonstrate how to use Evidently for monitoring machine learning models.

Let's proceed to retrieve the "Tutorials and guides" file. This step is necessary to gather detailed code examples that ca

In [25]:
from pydantic_ai import RunUsage

In [26]:
messages = []
usage = RunUsage()

while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    result = await search_agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage()

    print_messages(result.new_messages())
    messages.extend(result.new_messages())

request
USER: 

response
It appears that your message is empty. Please provide your question or request so I can assist you effectively.

request
USER: show me the code 

response
1) I will start with a search query using the term "code" to find relevant documentation that may contain code examples or snippets. This search is appropriate because the user is asking for code, so I want to locate any documents that might contain relevant programming or implementation details.

```json
{
  "query": "code"
}
```

2) Now, I will analyze the results from the search to see if there are relevant documents that contain code examples. Depending on the information available, I will refine my search or retrieve specific files that seem to contain relevant code documentation. 

Let's perform the search.
TOOL CALL: search {"query":"code"}

request
TOOL RETURN: search

response
2) The search results returned several documents that likely contain code examples or details:

- **Synthetic data generation

In [27]:
usage

RunUsage(input_tokens=3939, cache_read_tokens=1024, output_tokens=786, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, requests=4, tool_calls=3)

In [28]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

In [29]:
callback = NamedCallback(search_agent) 

In [30]:
result = await search_agent.run(
    "how do I build dashboads with evidently?",
    event_stream_handler=callback
)

TOOL CALL (search): search({"query":"build dashboards"})
TOOL CALL (search): get_file({"filename": "docs/platform/dashboard_add_panels.mdx"})
TOOL CALL (search): get_file({"filename": "docs/platform/dashboard_overview.mdx"})
